# SEAS the Moment — STAI-X 2026 Submission

Self-contained Colab notebook. **Runtime → Change runtime type → A100 GPU**, then **Run All**.

Before running: upload **`staix-challenge.zip`** (your data bundle: `train/`, `val/`, `images/`, `sample_submission.csv`) to `/content/` via the file panel on the left.

**What it does:** runs all four expert pipelines (Lenny FT-Transformer · William classical stats · Jasmine LightGBM · Eddy XGBoost), each in its own subprocess, then combines them per-category with inverse-MAE weights and the `all_drugs ≥ all_opioids/all_stimulants` nesting constraint → `submission.csv`.

Runtime ≈ 35–45 min (Lenny's transformer dominates).

## 1. Setup — clone repo + install

In [ ]:
!git clone https://github.com/lennardpische/staix26_seasthemoment.git
%cd staix26_seasthemoment
!git checkout moe-integration
!pip install -q -r requirements.txt

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only (Lenny will be slow)')

## 2. Unzip data into the repo root

In [ ]:
import zipfile, os

ZIP = '/content/staix-challenge.zip'   # ← upload this to /content/ first
assert os.path.exists(ZIP), f'Upload your data bundle to {ZIP} (file panel on the left)'
with zipfile.ZipFile(ZIP) as z:
    z.extractall('.')

for p in ['train/dose_sys_train.csv', 'train/covariates.csv', 'val/covariates.csv',
          'sample_submission.csv', 'period_id_map.json',
          'train/images/mat_density', 'val/images/mat_density']:
    print('✓' if os.path.exists(p) else '✗ MISSING', p)

## 3. (Optional) Pre-compute Lenny's embeddings
Gives the transformer real mpnet (768-d) text + EfficientNet (1408-d) image features instead of the TF-IDF/PCA fallback. ~10 min. Skip this cell and the pipeline still runs with the fallback.

In [ ]:
import sys
sys.path.insert(0, 'experts')
try:
    from lenny import vectorize
    vectorize.run()
    print('✓ embeddings/ cache built')
except Exception as e:
    print(f'Skipping embeddings ({e}) — transformer will use TF-IDF/PCA fallback')

## 4. Configure paths

In [ ]:
import sys, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd()
assert (REPO / 'experts').exists(), f'Run setup first — no experts/ in {REPO}'

def _find_data_root():
    kaggle = Path('/kaggle/input/staix-challenge')
    if kaggle.exists():
        return kaggle
    for p in [REPO, REPO / 'data']:
        if (p / 'train' / 'dose_sys_train.csv').exists():
            return p
    raise FileNotFoundError('No train/dose_sys_train.csv — did the unzip run?')

DATA_ROOT = _find_data_root()
TEMPLATE  = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
SCORING_CATEGORIES = ['all_drugs', 'all_opioids', 'all_stimulants']
EXPERTS = {n: REPO / 'experts' / n / 'run_expert.py'
           for n in ['lenny', 'william', 'jasmine', 'eddy']}
print(f'Repo: {REPO}')
print(f'Data: {DATA_ROOT}   Template: {len(TEMPLATE)} rows')

## 5. Run all four experts
Each runs in its own subprocess (so their identically-named modules never collide). The long one is Lenny (~25 min; resumes from `checkpoints/` if present). A failing expert is reported and skipped — the others still combine.

In [ ]:
def run_expert(name, runner):
    print(f'\n{"="*70}\n\u25b6 {name}\n{"="*70}', flush=True)
    t0 = time.time()
    p = subprocess.run(
        [sys.executable, str(runner), str(DATA_ROOT), f'expert_{name}.csv'],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(f'\u2717 {name} FAILED (exit {p.returncode}):\n{p.stderr[-3000:]}')
        return False
    print(f'\u2713 {name} done in {time.time()-t0:.0f}s')
    return True

status = {n: run_expert(n, r) for n, r in EXPERTS.items()}
print('\nStatus:', status)

## 6. Combine experts → submission.csv
Per-category inverse-MAE weighted average + nesting constraint.

⚠ Jasmine's ~0.2 MAEs come from target leakage (her rolling/EWMA features include the current period's value), so they over-state her skill and would give her ~80% of the weight. Until she fixes it, set her three `OOF_MAES` entries to `None` for equal weight.

In [ ]:
preds = {}
for name in EXPERTS:
    f = REPO / f'expert_{name}.csv'
    if not f.exists():
        print(f'  {name}: MISSING'); continue
    preds[name] = (pd.read_csv(f).set_index('row_id')['rate_per_10000_ed_visits']
                   .reindex(TEMPLATE['row_id'].values))
available = list(preds)
assert available, 'No expert CSVs produced'
print('Combining:', available)

OOF_MAES = {
    'lenny':   {'all_drugs': 3.0651, 'all_opioids': 1.3563, 'all_stimulants': 0.6995},
    'william': {'all_drugs': 2.9066, 'all_opioids': 1.3319, 'all_stimulants': 0.6664},
    'jasmine': {'all_drugs': 0.3705, 'all_opioids': 0.1725, 'all_stimulants': 0.1023},
    'eddy':    {'all_drugs': 2.9711, 'all_opioids': 1.3628, 'all_stimulants': 0.6907},
}

final = pd.Series(0.0, index=TEMPLATE['row_id'])
for cat in SCORING_CATEGORIES:
    ids = TEMPLATE.loc[TEMPLATE['overdose_category'] == cat, 'row_id'].values
    mats, w = [], []
    for n in available:
        mats.append(preds[n].reindex(ids).fillna(preds[n].median()).values)
        mae = OOF_MAES.get(n, {}).get(cat)
        w.append(1.0 / mae if mae else 1.0)
    w = np.array(w); w = w / w.sum()
    final.loc[ids] = (np.column_stack(mats) @ w).clip(0)
    print(f'  {cat:15s} ' + '  '.join(f'{n}={wi:.3f}' for n, wi in zip(available, w)))

# Nesting: all_drugs >= all_opioids and >= all_stimulants
d = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_drugs',      'row_id'].values
o = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_opioids',    'row_id'].values
s = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_stimulants', 'row_id'].values
final.loc[d] = np.maximum(np.maximum(final.loc[d].values, final.loc[o].values), final.loc[s].values)

submission = TEMPLATE[['row_id']].copy()
submission['rate_per_10000_ed_visits'] = final.reindex(TEMPLATE['row_id']).values
assert submission['rate_per_10000_ed_visits'].notna().all(), 'NaN in submission'
assert (submission['rate_per_10000_ed_visits'] >= 0).all(), 'Negative predictions'
assert set(submission['row_id']) == set(TEMPLATE['row_id']), 'row_id mismatch'

submission.to_csv('submission.csv', index=False)
print(f'\n\u2713 submission.csv written: {len(submission)} rows')
for cat in SCORING_CATEGORIES:
    v = submission.loc[(TEMPLATE['overdose_category'] == cat).values, 'rate_per_10000_ed_visits']
    print(f'  {cat:15s} mean={v.mean():.3f}  std={v.std():.3f}  min={v.min():.3f}  max={v.max():.3f}')
submission.head()

## 7. Download

In [ ]:
try:
    from google.colab import files
    files.download('submission.csv')
except Exception as e:
    print('Download manually from the file panel —', e)